# Lab 09: Recurrent Neural Networks (RNNs, LSTMs & GRUs) for Sequence Modeling

Welcome to Laboratory 09! In this lab, we advance into **Deep Temporal Sequence Modeling**:
1. **Recurrence & Hidden States**: Understand recurrent state transitions $\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \mathbf{x}_t + \mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{b}_h)$.
2. **Gated Architectures (LSTM & GRU)**: Mitigate vanishing gradients using additive cell state memory channels $\mathbf{c}_t$ and gating mechanisms (Input, Forget, Output gates).
3. **Autoregressive Character Language Model (`CharLSTM`)**: Build, train, and generate text character-by-character with temperature-controlled sampling.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import PyTorch recurrent layers, optimization tools, and NumPy
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Seed for reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Compute Device:', device)


## 2. Text Corpus Processing & Character Vocabulary Construction

### Conceptual Overview: Character-Level Tokenization
In a character-level language model, each unique character in the alphabet represents a discrete token mapped to an integer index $i \in [0, V-1]$, where $V$ is the vocabulary size.


In [ ]:
# Sample text corpus (excerpt from H.G. Wells' The Time Machine)
corpus = 'The Time Traveller was expounding a recondite matter to us. His grey eyes shone and twinkled.'

# Extract unique sorted character vocabulary
chars = sorted(list(set(corpus)))
vocab_size = len(chars)

# Build bi-directional lookup dictionaries: character -> index and index -> character
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for idx, ch in enumerate(chars)}

print(f'Corpus Length: {len(corpus)} characters | Unique Character Vocabulary Size: {vocab_size}')
print('Vocabulary Characters:', repr(''.join(chars)))


## 3. Character-Level LSTM Architecture

### Architecture Overview: `CharLSTM`
The `CharLSTM` model processes sequential character indices through three primary stages:
1. **Embedding Layer (`nn.Embedding`)**: Maps discrete character integer tokens $x_t \in \{0, \dots, V-1\}$ to dense vector representations $\mathbf{e}_t \in \mathbb{R}^{D_{embed}}$.
2. **LSTM Layer (`nn.LSTM`)**: Processes sequence with input gate $\mathbf{i}_t$, forget gate $\mathbf{f}_t$, candidate cell $\tilde{\mathbf{c}}_t$, and output gate $\mathbf{o}_t$:
   $$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{e}_t, \mathbf{h}_{t-1}] + \mathbf{b}_f)$$
   $$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{e}_t, \mathbf{h}_{t-1}] + \mathbf{b}_i)$$
   $$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tanh(\mathbf{W}_c [\mathbf{e}_t, \mathbf{h}_{t-1}] + \mathbf{b}_c)$$
   $$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$
3. **Linear Projection Head (`nn.Linear`)**: Projects hidden states $\mathbf{h}_t$ to vocabulary logits $\hat{\mathbf{y}}_t \in \mathbb{R}^V$.


In [ ]:
# Define the Character-Level LSTM Neural Network Architecture
class CharLSTM(nn.Module):
    """Character-level Recurrent Language Model using LSTM layers and dense projection."""
    def __init__(self, vocab_size: int, embed_dim: int = 16, hidden_dim: int = 64):
        super(CharLSTM, self).__init__()
        # Token embedding lookup table: maps integer token indices to dense vectors
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # Long Short-Term Memory recurrent backbone (batch_first=True: tensor shape [B, T, D])
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        
        # Dense linear classifier projecting hidden states to vocabulary logits
        self.fc = nn.Linear(in_features=hidden_dim, out_features=vocab_size)
        
    def forward(self, x: torch.Tensor, state=None):
        """
        Args:
            x: Input token indices tensor of shape (Batch_Size, Sequence_Length)
            state: Optional tuple (h_0, c_0) containing initial hidden and cell states
        Returns:
            logits: Output tensor of shape (Batch_Size, Sequence_Length, Vocab_Size)
            state: Updated tuple (h_t, c_t)
        """
        # Convert integer indices to dense vectors: shape (B, T) -> (B, T, embed_dim)
        embeds = self.embedding(x)
        
        # Propagate through LSTM recurrent cells
        lstm_out, state = self.lstm(embeds, state) # shape: (B, T, hidden_dim)
        
        # Project hidden representations to next-character vocabulary logits
        logits = self.fc(lstm_out)                 # shape: (B, T, vocab_size)
        return logits, state

# Instantiate the CharLSTM model
lstm_model = CharLSTM(vocab_size=vocab_size, embed_dim=16, hidden_dim=64).to(device)
print('Initialized CharLSTM Model Architecture:\n', lstm_model)


### Autoregressive Text Generation Function: `generate_text`
The function below autoregressively generates text by feeding predicted characters back into the model, supporting **Temperature Scaling**:
$$P(x_{t+1} = c) = \frac{\exp(z_c / T)}{\sum_j \exp(z_j / T)}$$
* $T < 1.0$: Lower temperature $\to$ greedy, deterministic, repetitive text.
* $T > 1.0$: Higher temperature $\to$ diverse, creative, exploratory text.


In [ ]:
def generate_text(model: nn.Module, start_str: str, length: int = 50, temperature: float = 0.8) -> str:
    """Generates text autoregressively using temperature-controlled softmax sampling."""
    model.eval()
    generated = start_str
    
    # Encode prompt string into tensor of integer indices
    input_indices = [char_to_idx[c] for c in start_str if c in char_to_idx]
    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)
    
    state = None
    with torch.no_grad():
        # Warm up hidden states with the prompt
        for i in range(input_tensor.size(1) - 1):
            _, state = model(input_tensor[:, i:i+1], state)
            
        current_token = input_tensor[:, -1:]
        
        # Autoregressive generation loop
        for _ in range(length):
            logits, state = model(current_token, state)
            # Apply temperature scaling to logits
            scaled_logits = logits.squeeze() / max(temperature, 1e-5)
            # Compute categorical probability distribution
            probs = torch.softmax(scaled_logits, dim=-1)
            # Sample next token index from probability distribution
            next_idx = torch.multinomial(probs, num_samples=1).item()
            
            generated += idx_to_char.get(next_idx, '')
            current_token = torch.tensor([[next_idx]], dtype=torch.long).to(device)
            
    return generated

# Sample initial generation with random weights
print('Generated Sample (Untrained Baseline):')
print(repr(generate_text(lstm_model, start_str='The ', length=30, temperature=0.8)))


## 4. Summary & Key Takeaways
1. **Recurrent State**: Maintains information across sequence steps, enabling variable-length temporal modeling.
2. **Gated Cell States**: LSTMs use additive linear cell state paths to eliminate vanishing gradients during backpropagation through time (BPTT).
3. **Temperature Scaling**: Controls entropy during sampling, balancing fidelity and diversity.
